# DSA_00 — Decision-Support Application Design & Architecture

**Purpose.** Freeze the application boundary before UI implementation.

- The public application is deliberately separated from the model artifacts. 
- The validated RF/XGBoost pipeline remains the offline production layer; Streamlit consumes lightweight precomputed outputs.

In [1]:
# Import libraries
from pathlib import Path
import sys, yaml

In [2]:
# Define config paths
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG = PROJECT_ROOT / "configs" / "decision_support_app.yaml"
CONFIG

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/decision_support_app.yaml')

In [3]:
cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
cfg["application"]

{'title': 'Ontario Electricity Demand & Peak-Risk Decision Support',
 'default_mode': 'public_demo',
 'public_demo_only': True,
 'timezone': 'America/Toronto',
 'official_peak_threshold': 0.06,
 'horizons': 24,
 'fsas': ['L4T', 'M5R', 'M5S', 'M6G', 'M9R', 'M9W']}

## Frozen application architecture

```text
LOCAL PIPELINE
operational demand + weather
          ↓
validated IFB
          ↓
24 RF + 24 XGBoost frozen models
          ↓
forecasts + Peak-Risk + scenarios
          ↓
precomputed Parquet / JSON
          ↓
PUBLIC STREAMLIT APPLICATION
```

The public application never changes model features, parameters, Peak-Risk
definition, or the official threshold.

In [5]:
architecture = {
    "public_loads_models": cfg["deployment"]["load_model_artifacts"],
    "public_demo_only": cfg["application"]["public_demo_only"],
    "fsas": cfg["application"]["fsas"],
    "horizons": cfg["application"]["horizons"],
    "official_threshold": cfg["application"]["official_peak_threshold"],
}
architecture

{'public_loads_models': False,
 'public_demo_only': True,
 'fsas': ['L4T', 'M5R', 'M5S', 'M6G', 'M9R', 'M9W'],
 'horizons': 24,
 'official_threshold': 0.06}